# 🌿 Getting Started with BioCLIP 2

**BioCLIP 2** is a vision foundation model trained on millions of biological images spanning the tree of life. It can classify organisms from photos — down to the species level — using the full Linnaean taxonomic hierarchy.

**pybioclip** is a Python package that makes it easy to use BioCLIP 2 without any ML background.

In this tutorial you will:
1. Run a pre-loaded demo with sample images
2. Explore predictions at different taxonomic ranks
3. Try custom label classification
4. **Bring your own images** and explore on your own

---
> 📋 **Citation:** If you use BioCLIP 2 in research, cite [Gu et al. (NeurIPS 2025)](https://openreview.net/forum?id=yPC9zmkQgG) and [pybioclip](https://doi.org/10.5281/zenodo.13151194).
>
> 🔗 **Docs:** [imageomics.github.io/pybioclip](https://imageomics.github.io/pybioclip/)

## 📦 Step 1: Install and import

In [ ]:
!pip install pybioclip ipywidgets --quiet
print("✅ Done!")

In [ ]:
import os
import shutil
import requests
import pandas as pd
import PIL.Image
import matplotlib.pyplot as plt
from bioclip import TreeOfLifeClassifier, Rank

## 🖼️ Step 2: Load demo images

We'll start with three sample images from the [BioCLIP demo](https://huggingface.co/spaces/imageomics/bioclip-demo): a bear, a sea anemone, and a cat.

In [ ]:
IMAGE_URLS = [
    'https://huggingface.co/spaces/imageomics/bioclip-demo/resolve/main/examples/Ursus-arctos.jpeg',
    'https://huggingface.co/spaces/imageomics/bioclip-demo/resolve/main/examples/Actinostola-abyssorum.png',
    'https://huggingface.co/spaces/imageomics/bioclip-demo/resolve/main/examples/Felis-catus.jpeg',
]

IMAGE_DIR = "images"
THUMBNAIL_SIZE = (300, 300)

def download_image(url):
    filename = url.split('/')[-1].split('?')[0]
    os.makedirs(IMAGE_DIR, exist_ok=True)
    path = os.path.join(IMAGE_DIR, filename)
    headers = {'User-Agent': 'Mozilla/5.0 (compatible; BioCLIP-tutorial/1.0)'}
    response = requests.get(url, stream=True, headers=headers)
    response.raise_for_status()
    with open(path, 'wb') as f:
        shutil.copyfileobj(response.raw, f)
    return path

print("Downloading images...")
image_paths = [download_image(url) for url in IMAGE_URLS]
print("Downloaded:", image_paths)

fig, axes = plt.subplots(1, len(image_paths), figsize=(4 * len(image_paths), 4))
for ax, path in zip(axes, image_paths):
    img = PIL.Image.open(path)
    img.thumbnail(THUMBNAIL_SIZE)
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(os.path.basename(path), fontsize=8)
plt.suptitle('Demo images', fontsize=12)
plt.tight_layout()
plt.show()

## 🌳 Step 3: Predict species using the Tree of Life

`TreeOfLifeClassifier` ranks each image against BioCLIP 2's full taxonomic database — hundreds of thousands of species.

> ⚠️ **Note:** You may see warnings about `HF_TOKEN` and `os.fork()` — these are harmless.

In [ ]:
classifier = TreeOfLifeClassifier(device='cpu')
predictions = classifier.predict(image_paths, rank=Rank.SPECIES)

df = pd.DataFrame(predictions)
df

### Top prediction per image

In [ ]:
top1 = df.loc[df.groupby('file_name')['score'].idxmax()].reset_index(drop=True)
top1['score_%'] = (top1['score'] * 100).round(1).astype(str) + '%'
display(top1[['file_name', 'species', 'common_name', 'score_%']])

### Visualize top 5 predictions for one image

In [ ]:
example_path = image_paths[0]  # bear

single = df[df['file_name'] == example_path].head(5)
labels = [
    f"{r['species']}\n({r['common_name']})" if r['common_name'] else r['species']
    for _, r in single.iterrows()
]
scores = single['score'].tolist()
colors = ['#2a9d8f'] + ['#a8dadc'] * (len(scores) - 1)

fig, (ax_img, ax_bar) = plt.subplots(1, 2, figsize=(12, 4),
                                      gridspec_kw={'width_ratios': [1, 2]})
img = PIL.Image.open(example_path)
img.thumbnail(THUMBNAIL_SIZE)
ax_img.imshow(img)
ax_img.axis('off')
ax_img.set_title(os.path.basename(example_path), fontsize=9)

bars = ax_bar.barh(labels[::-1], scores[::-1], color=colors[::-1], edgecolor='white')
ax_bar.set_xlabel('Confidence score')
ax_bar.set_title('BioCLIP 2 — Top 5 Species Predictions')
ax_bar.set_xlim(0, 1)
for bar, score in zip(bars, scores[::-1]):
    ax_bar.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
                f'{score*100:.1f}%', va='center', fontsize=9)
plt.tight_layout()
plt.show()

## 🏷️ Step 4: Custom label classification

`CustomLabelsClassifier` lets you define your own categories — useful when you only care about a specific set of organisms.

In [ ]:
from bioclip import CustomLabelsClassifier

my_labels = ['duck', 'fish', 'bear']
example_path = image_paths[0]  # bear image

custom_classifier = CustomLabelsClassifier(my_labels)
custom_preds = custom_classifier.predict(example_path)

print(f"Image: {os.path.basename(example_path)}")
print(f"Labels: {my_labels}\n")
for p in custom_preds:
    bar = '█' * int(p['score'] * 30)
    print(f"  {p['classification']:15s}  {bar}  {p['score']*100:.2f}%")

---
# 🧪 Your Turn: Exploration Exercises

Now it's your turn to experiment. Each exercise below has blanks (`___`) for you to fill in.
Run each cell after completing it to see your results.

**Finding image URLs:**
- Go to [Wikimedia Commons](https://commons.wikimedia.org/) and search for any organism
- Click on an image, then right-click the image → **Copy image address**
- Or use [iNaturalist](https://www.inaturalist.org/) — right-click any observation photo → Copy image address

## Exercise 1: Load your own image

Find an image URL of any organism you're interested in and load it.

In [ ]:
# ✏️ Replace the URL with one of your own
my_url = '___'

my_path = download_image(my_url)

img = PIL.Image.open(my_path)
img.thumbnail(THUMBNAIL_SIZE)
plt.imshow(img)
plt.axis('off')
plt.title(os.path.basename(my_path))
plt.show()

## Exercise 2: Predict the species

Run the Tree of Life classifier on your image and look at the top results.

In [ ]:
# ✏️ Fill in the blank — what rank do you want to predict at?
# Options: Rank.KINGDOM, Rank.PHYLUM, Rank.CLASS, Rank.ORDER,
#          Rank.FAMILY, Rank.GENUS, Rank.SPECIES
my_rank = Rank.___

my_predictions = classifier.predict([my_path], rank=my_rank)
my_df = pd.DataFrame(my_predictions)
my_df.head(10)

## Exercise 3: Change the rank

Try predicting at a broader taxonomic level, like `Rank.GENUS` or `Rank.ORDER`. How do the results change?

In [ ]:
# ✏️ Try a different rank than you used above
broader_rank = Rank.___

broader_predictions = classifier.predict([my_path], rank=broader_rank)
broader_df = pd.DataFrame(broader_predictions)
broader_df.head(10)

## Exercise 4: Custom labels

Create your own set of labels relevant to the organism in your image and see which one scores highest.

In [ ]:
# ✏️ Fill in labels that make sense for your image
# Example: ['salmon', 'trout', 'bass', 'carp', 'catfish']
my_labels = ['___', '___', '___']

my_custom_classifier = CustomLabelsClassifier(my_labels)
my_custom_preds = my_custom_classifier.predict(my_path)

print(f"Image: {os.path.basename(my_path)}")
print(f"Labels: {my_labels}\n")
for p in my_custom_preds:
    bar = '█' * int(p['score'] * 30)
    print(f"  {p['classification']:20s}  {bar}  {p['score']*100:.2f}%")

## Exercise 5 (Bonus): Compare two images

Find a second image URL and classify both at once. Do the top predictions make sense?

In [ ]:
# ✏️ Add a second image URL
my_url_2 = '___'

my_path_2 = download_image(my_url_2)
my_two_paths = [my_path, my_path_2]

# Predict on both
two_predictions = classifier.predict(my_two_paths, rank=Rank.SPECIES)
two_df = pd.DataFrame(two_predictions)

# Show top result for each
top = two_df.loc[two_df.groupby('file_name')['score'].idxmax()].reset_index(drop=True)
top['score_%'] = (top['score'] * 100).round(1).astype(str) + '%'
display(top[['file_name', 'species', 'common_name', 'score_%']])

# Show both images
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for ax, path in zip(axes, my_two_paths):
    img = PIL.Image.open(path)
    img.thumbnail(THUMBNAIL_SIZE)
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(os.path.basename(path), fontsize=8)
plt.tight_layout()
plt.show()

---
## 🧹 Cleanup

In [ ]:
shutil.rmtree(IMAGE_DIR, ignore_errors=True)
print('🧹 Temp files removed.')

---
## 📚 Learn More

- **[Different taxonomic ranks](https://imageomics.github.io/pybioclip/python-tutorial/)** — full pybioclip documentation
- **[iNaturalist images](https://colab.research.google.com/github/Imageomics/pybioclip/blob/main/examples/iNaturalistPredict.ipynb)** — pull images directly from iNaturalist observations
- **[TreeOfLife subsetting](https://colab.research.google.com/github/Imageomics/pybioclip/blob/main/examples/TOL-Subsetting.ipynb)** — filter predictions to a taxonomic group
- **[GradCAM explainability](https://colab.research.google.com/github/Imageomics/pybioclip/blob/main/examples/GradCamExperiment.ipynb)** — visualize which image regions drive predictions
- **[Few-shot classifiers](https://imageomics.github.io/pybioclip/python-tutorial/#lightweight-classifiers)** — SVM/Ridge/SimpleShot on top of BioCLIP embeddings